# 09 — M3OT geliştirme bölümü: dönüşüm ve paketler (Faz 3 Görev 3-4)

Çalışma zamanı → **CPU** (GPU gerekmez). Secrets: `GITHUB_TOKEN`. Kaggle gerekmez.
**Tümünü çalıştır değil:** hücre 1-5'i sırayla çalıştır, hücre 5'ten sonra dur (KAPI).

⚠️ **Test dizileri (1-03, 2-03) yalnız paketlenir; ölçüm yok.**
⚠️ **Hücre 3'ün çıktısını paylaşmak güvenli: cURL komutu ve çerezler hiçbir yere yazılmaz.**

İndirme adımı: figshare sayfasında (docs/data/m3ot.md'deki bağlantı) indirmeyi başlat, tarayıcı
geliştirici araçları Ağ sekmesinde indirme isteğine sağ tık → 'Copy as cURL (bash)' → hücre 3
sorduğunda yapıştır (çok satırlı olması sorun değil, hücre birleştirir); çerezler kısa ömürlü, her indirmede yeniden kopyalanır.
Toplam süre ≈ 30-45 dk.

In [ ]:
# ≈ 1 dk
BRANCH = 'f3'
DRIVE = '/content/drive/MyDrive/baykus'
RAW = '/content/m3ot_raw'
OUT = '/content/m3ot'
ZIP = '/content/M3OT.zip'
import os
NPROC = os.cpu_count()
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(f'{DRIVE}/m3ot', exist_ok=True)

In [ ]:
%%time
# ≈ 2 dk
import os
from google.colab import userdata
try:
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = f'https://{tok}@github.com/OgzhnTarhn/baykus.git' if tok else 'https://github.com/OgzhnTarhn/baykus.git'
if not os.path.isdir('/content/baykus/.git'):
    !git clone -q {url} /content/baykus
!git -C /content/baykus fetch -q {url} {BRANCH}
!git -C /content/baykus checkout -q -B {BRANCH} FETCH_HEAD
!git -C /content/baykus remote set-url origin https://github.com/OgzhnTarhn/baykus.git
%cd /content/baykus
!git log --oneline -3
!pip -q install uv
!nproc
!uv pip install --system -q -e "/content/baykus/python"
import sys
sys.path.insert(0, '/content/baykus/python/src')

In [ ]:
%%time
# ≈ 5-15 dk, 10 GB
import os
import re
import shlex
from getpass import getpass

from baykus.data.secret import run_secret

if os.path.exists(ZIP) and os.path.getsize(ZIP) >= 9e9:
    print(f'M3OT.zip zaten var ({os.path.getsize(ZIP) / 1e9:.2f} GB), indirme atlandı')
else:
    cmd = getpass('figshare cURL komutu: ')
    try:
        # Chrome 'Copy as cURL (bash)' çok satırlı: yapıştırınca '\' + boşluklar kalır → tek satır.
        # $'...' tırnakları için bash ile çalışır (run_secret /bin/sh kullanır).
        cmd = re.sub(r'\\\s+', ' ', cmd).strip()
        rc = run_secret('bash -c ' + shlex.quote(cmd + f' -o {ZIP} -sS --fail'), label='M3OT indirme')
    finally:
        del cmd
    if rc != 0:
        raise RuntimeError('M3OT indirme başarısız: cURL komutunu tarayıcıdan yeniden kopyala')
    if not os.path.exists(ZIP) or os.path.getsize(ZIP) < 9e9:
        size = os.path.getsize(ZIP) / 1e9 if os.path.exists(ZIP) else 0.0
        raise RuntimeError(f'M3OT.zip eksik ya da küçük ({size:.2f} GB): HTML hata sayfası inmiş olabilir; cURL\'ü yeniden kopyala')
    print(f'M3OT.zip: {os.path.getsize(ZIP) / 1e9:.2f} GB')

In [ ]:
%%time
# ≈ 10-15 dk
!unzip -q -n {ZIP} -d {RAW} && rm -f {ZIP}
!du -sh {RAW}
!df -h /content
!baykus-data m3ot --raw {RAW} --out {OUT} --splits val train test --workers {NPROC}
!cp {OUT}/meta/m3ot_stats.json {DRIVE}/m3ot/m3ot_stats.json
!cat {OUT}/meta/m3ot_stats.json
!du -sh {OUT}/*

In [ ]:
%%time
# ≈ 1 dk · KAPI: bu hücrenin çıktısının TAMAMINI Claude'a getir
for s in ['val', 'train', 'test']:
    print(f'=== {s} ===')
    !baykus-data m3ot-seqs --root {OUT} --split {s} | tee {DRIVE}/m3ot/seqs_{s}.json

🚧 **KAPI:** Burada dur. Hücre 4 ve 5'in çıktısını Claude'a getir. Claude geliştirme bölümünü (DEV_SPLIT) ve dizileri (DEV_SEQS) verir; hücre 7'ye elle gir, sonra hücre 7'yi çalıştır.

In [ ]:
%%time
# ≈ 3-5 dk
DEV_SPLIT = None  # Claude'un KAPI'da verdiği değer ('val' ya da 'train')
DEV_SEQS = []     # Claude'un KAPI'da verdiği diziler
if DEV_SPLIT not in ('val', 'train') or not DEV_SEQS:
    raise ValueError('DEV_SPLIT val/train olmalı ve DEV_SEQS boş olmamalı (KAPI çıktısına göre doldur)')
seqs = ' '.join(DEV_SEQS)
!rm -rf /content/dev_pack /content/test_pack
!baykus-data demo-pack --root {OUT} --split {DEV_SPLIT} --seqs {seqs} --out /content/dev_pack
# test yalnız paketlenir, ölçüm Görev 36'da; demo-pack etiketleri de (pack_labels) yazar
!baykus-data demo-pack --root {OUT} --split test --seqs 1-03 2-03 --allow-test --out /content/test_pack
!rm -f {DRIVE}/m3ot/dev_pack.zip {DRIVE}/m3ot/test_pack.zip
!cd /content/dev_pack && zip -q -r -0 {DRIVE}/m3ot/dev_pack.zip sequences labels
!cd /content/test_pack && zip -q -r -0 {DRIVE}/m3ot/test_pack.zip sequences labels
!du -sh /content/dev_pack /content/test_pack
!ls -lh {DRIVE}/m3ot/dev_pack.zip {DRIVE}/m3ot/test_pack.zip